In [ ]:
import pandas as pd

df = pd.read_csv(
    '../data/food.csv.gz',
    sep='\t',                  # TAB separated, not comma!
    compression='gzip',
    nrows=500000,              # Only load first 500k rows
    low_memory=False
)

print(df.shape)
print(df.columns.tolist())

In [ ]:
# Find our key columns
sugar_cols = [col for col in df.columns if 'sugar' in col.lower()]
protein_cols = [col for col in df.columns if 'protein' in col.lower()]
fat_cols = [col for col in df.columns if 'fat' in col.lower()]
fiber_cols = [col for col in df.columns if 'fiber' in col.lower()]
category_cols = [col for col in df.columns if 'categor' in col.lower()]

print("Sugar columns:", sugar_cols)
print("Protein columns:", protein_cols)
print("Fat columns:", fat_cols)
print("Fiber columns:", fiber_cols)
print("Category columns:", category_cols)

In [ ]:
# Preview the key columns we expect to use
key_cols = ['product_name', 'categories_tags', 'ingredients_text']
print(df[key_cols].head(3))

In [ ]:
# Check how much data is missing in our key columns
key_cols = ['product_name', 'categories_tags', 'ingredients_text', 
            'sugars_100g', 'proteins_100g', 'fat_100g', 'fiber_100g']

missing = df[key_cols].isnull().sum()
total = len(df)

print("Missing values:\n")
for col, count in missing.items():
    pct = (count / total) * 100
    print(f"  {col}: {count} missing ({pct:.1f}%)")

In [ ]:
print("Shape before cleaning:", df.shape)

cols_to_keep = [
    'product_name',
    'categories_tags',
    'ingredients_text',
    'sugars_100g',
    'proteins_100g',
    'fat_100g',
    'fiber_100g'
]
df_clean = df[cols_to_keep].copy()

df_clean = df_clean.dropna(subset=['product_name', 'sugars_100g', 'proteins_100g'])

print("Shape after dropping missing essentials:", df_clean.shape)

df_clean = df_clean[
    (df_clean['sugars_100g'] >= 0) & (df_clean['sugars_100g'] <= 100) &
    (df_clean['proteins_100g'] >= 0) & (df_clean['proteins_100g'] <= 100) &
    (df_clean['fat_100g'].isna() | ((df_clean['fat_100g'] >= 0) & (df_clean['fat_100g'] <= 100)))
]

print("Shape after removing impossible values:", df_clean.shape)

df_clean = df_clean.reset_index(drop=True)

print("\n Cleaning complete!")
print("Final clean dataset shape:", df_clean.shape)
print("\nMissing values in clean dataset:")
print(df_clean.isnull().sum())

In [ ]:
# Preview categories_tags to understand what we're working with
print(df_clean['categories_tags'].dropna().head(10).tolist())

In [ ]:
# ============================================
# STORY 2: IMPROVED CATEGORY GROUPING
# ============================================

def assign_category(tags):
    if not isinstance(tags, str):
        return 'Other'
    
    tags = tags.lower()
    
    # Order matters — more specific first!
    if any(k in tags for k in ['chocolate', 'candy', 'confectionery', 'sweet-snack', 'sugar-confectionery']):
        return 'Confectionery'
    elif any(k in tags for k in ['biscuit', 'cookie', 'cake', 'pastry', 'doughnut', 'wafer', 'muffin', 'brownie']):
        return 'Biscuits & Cakes'
    elif any(k in tags for k in ['snack', 'chip', 'crisp', 'popcorn', 'pretzel', 'cracker', 'puff']):
        return 'Snacks'
    elif any(k in tags for k in ['cereal', 'breakfast', 'granola', 'muesli', 'oat', 'porridge']):
        return 'Cereals & Breakfast'
    elif any(k in tags for k in ['bread', 'loaf', 'toast', 'bagel', 'roll', 'wrap', 'bakery', 'baked']):
        return 'Bread & Bakery'
    elif any(k in tags for k in ['spread', 'jam', 'honey', 'hazelnut', 'peanut-butter', 'marmalade']):
        return 'Spreads'
    elif any(k in tags for k in ['yogurt', 'yoghurt', 'cheese', 'milk', 'dairy', 'butter', 'cream']):
        return 'Dairy'
    elif any(k in tags for k in ['meat', 'chicken', 'beef', 'pork', 'sausage', 'chorizo', 'ham', 'bacon', 'poultry']):
        return 'Meat & Poultry'
    elif any(k in tags for k in ['seafood', 'fish', 'tuna', 'salmon', 'shrimp', 'prawn']):
        return 'Seafood'
    elif any(k in tags for k in ['beverage', 'drink', 'juice', 'soda', 'water', 'cola', 'tea', 'coffee', 'smoothie', 'milk-based-beverage']):
        return 'Beverages'
    elif any(k in tags for k in ['sauce', 'condiment', 'spice', 'seasoning', 'dressing', 'vinegar', 'ketchup', 'mustard']):
        return 'Sauces & Condiments'
    elif any(k in tags for k in ['vegetable', 'fruit', 'plant-based', 'vegan', 'legume', 'bean', 'lentil', 'tofu', 'nuts']):
        return 'Plant Based'
    elif any(k in tags for k in ['pasta', 'rice', 'noodle', 'grain', 'flour', 'starch']):
        return 'Grains & Pasta'
    elif any(k in tags for k in ['frozen', 'ready-meal', 'prepared', 'instant']):
        return 'Ready Meals'
    else:
        return 'Other'

# Apply improved function
df_clean['primary_category'] = df_clean['categories_tags'].apply(assign_category)

# Check the distribution
print("Category Distribution:")
print(df_clean['primary_category'].value_counts())
print(f"\nTotal categories: {df_clean['primary_category'].nunique()}")
print(f"\nOther percentage: {(df_clean['primary_category'] == 'Other').sum() / len(df_clean) * 100:.1f}%")

In [ ]:
# Let's peek inside the 'Other' bucket
other_sample = df_clean[df_clean['primary_category'] == 'Other']['categories_tags'].dropna().head(20).tolist()

for item in other_sample:
    print(item)
    print("---")

In [ ]:
def assign_category(tags):
    if not isinstance(tags, str):
        return 'Other'
    
    tags = tags.lower()
    
    if any(k in tags for k in ['protein-powder', 'protein-shake', 'bodybuilding', 'dietary-supplement', 'meal-replacement', 'vitamin', 'mineral']):
        return 'Supplements & Protein'
    elif any(k in tags for k in ['soup', 'broth', 'bouillon', 'potage']):
        return 'Soups'
    elif any(k in tags for k in ['chocolate', 'candy', 'confectionery', 'sweet-snack', 'sugar-confectionery']):
        return 'Confectionery'
    elif any(k in tags for k in ['biscuit', 'cookie', 'cake', 'pastry', 'doughnut', 'wafer', 'muffin', 'brownie']):
        return 'Biscuits & Cakes'
    elif any(k in tags for k in ['snack', 'chip', 'crisp', 'popcorn', 'pretzel', 'cracker', 'puff']):
        return 'Snacks'
    elif any(k in tags for k in ['cereal', 'breakfast', 'granola', 'muesli', 'oat', 'porridge']):
        return 'Cereals & Breakfast'
    elif any(k in tags for k in ['bread', 'loaf', 'toast', 'bagel', 'roll', 'wrap', 'bakery', 'baked']):
        return 'Bread & Bakery'
    elif any(k in tags for k in ['spread', 'jam', 'honey', 'hazelnut', 'peanut-butter', 'marmalade']):
        return 'Spreads'
    elif any(k in tags for k in ['yogurt', 'yoghurt', 'cheese', 'milk', 'dairy', 'butter', 'cream']):
        return 'Dairy'
    elif any(k in tags for k in ['meat', 'chicken', 'beef', 'pork', 'sausage', 'chorizo', 'ham', 'bacon', 'poultry']):
        return 'Meat & Poultry'
    elif any(k in tags for k in ['seafood', 'fish', 'tuna', 'salmon', 'shrimp', 'prawn']):
        return 'Seafood'
    elif any(k in tags for k in ['beverage', 'drink', 'juice', 'soda', 'water', 'cola', 'tea', 'coffee', 'smoothie']):
        return 'Beverages'
    elif any(k in tags for k in ['sauce', 'condiment', 'spice', 'seasoning', 'dressing', 'vinegar', 'ketchup', 'mustard']):
        return 'Sauces & Condiments'
    elif any(k in tags for k in ['vegetable', 'fruit', 'plant-based', 'vegan', 'legume', 'bean', 'lentil', 'tofu', 'nuts']):
        return 'Plant Based'
    elif any(k in tags for k in ['pasta', 'rice', 'noodle', 'grain', 'flour', 'starch']):
        return 'Grains & Pasta'
    elif any(k in tags for k in ['frozen', 'ready-meal', 'prepared', 'instant']):
        return 'Ready Meals'
    else:
        return 'Other'

# Apply final version
df_clean['primary_category'] = df_clean['categories_tags'].apply(assign_category)

# Results
print("Category Distribution:")
print(df_clean['primary_category'].value_counts())
print(f"\nTotal categories: {df_clean['primary_category'].nunique()}")
print(f"\nOther percentage: {(df_clean['primary_category'] == 'Other').sum() / len(df_clean) * 100:.1f}%")

In [ ]:
# How many 'Other' products have NO categories_tags at all?
other_df = df_clean[df_clean['primary_category'] == 'Other']

no_tags = other_df['categories_tags'].isna().sum()
has_tags = other_df['categories_tags'].notna().sum()

print(f"'Other' products with NO tags: {no_tags} ({no_tags/len(other_df)*100:.1f}%)")
print(f"'Other' products WITH tags: {has_tags} ({has_tags/len(other_df)*100:.1f}%)")

In [ ]:
# Drop 'Other' category - they have no useful category information
df_final = df_clean[df_clean['primary_category'] != 'Other'].copy()
df_final = df_final.reset_index(drop=True)

print("Final dataset shape:", df_final.shape)
print(f"\nCategory Distribution:")
print(df_final['primary_category'].value_counts())
print(f"\nTotal products for analysis: {len(df_final)}")

In [ ]:
# ============================================
# STORY 3: NUTRIENT MATRIX VISUALIZATION
# ============================================

import plotly.express as px
import pandas as pd


fig = px.scatter(
    df_final,
    x='sugars_100g',
    y='proteins_100g',
    color='primary_category',
    title='Sugar vs Protein — Market Gap Analysis (The Nutrient Matrix)',
    labels={
        'sugars_100g': 'Sugar per 100g (g)',
        'proteins_100g': 'Protein per 100g (g)',
        'primary_category': 'Category'
    },
    opacity=0.5,
    hover_data=['product_name']
)


sugar_avg = df_final['sugars_100g'].median()
protein_avg = df_final['proteins_100g'].median()

fig.add_hline(y=protein_avg, line_dash="dash", line_color="white",
              annotation_text=f"Avg Protein: {protein_avg:.1f}g")
fig.add_vline(x=sugar_avg, line_dash="dash", line_color="white",
              annotation_text=f"Avg Sugar: {sugar_avg:.1f}g")

fig.update_layout(
    width=1100,
    height=700,
    template='plotly_dark'
)

fig.show()

print(f"\nMedian Sugar: {sugar_avg:.2f}g")
print(f"Median Protein: {protein_avg:.2f}g")

In [ ]:
# ============================================
# STORY 4: THE RECOMMENDATION
# ============================================

# Define the "Blue Ocean" quadrant — High Protein + Low Sugar
blue_ocean = df_final[
    (df_final['sugars_100g'] < 5.0) &
    (df_final['proteins_100g'] > 6.67)
]

print(f"Total products in Blue Ocean quadrant: {len(blue_ocean)}")
print(f"That's only {len(blue_ocean)/len(df_final)*100:.1f}% of all products!\n")

# What categories dominate the Blue Ocean?
print("Categories in the Blue Ocean:")
print(blue_ocean['primary_category'].value_counts())

# What are the average nutrients in the Blue Ocean?
print(f"\nBlue Ocean Average Nutrients:")
print(f"  Average Protein: {blue_ocean['proteins_100g'].mean():.1f}g")
print(f"  Average Sugar: {blue_ocean['sugars_100g'].mean():.1f}g")

# What category has the LEAST products in Blue Ocean vs total?
print("\nGap Score (how underserved each category is in Blue Ocean):")
for cat in df_final['primary_category'].unique():
    total = len(df_final[df_final['primary_category'] == cat])
    in_blue_ocean = len(blue_ocean[blue_ocean['primary_category'] == cat])
    pct = (in_blue_ocean / total) * 100
    print(f"  {cat}: {in_blue_ocean}/{total} products in Blue Ocean ({pct:.1f}%)")

In [ ]:
# Clean Gap Score Table
print("Gap Score (sorted by LEAST served in Blue Ocean):")
print("-" * 60)

gap_data = []
for cat in df_final['primary_category'].unique():
    total = len(df_final[df_final['primary_category'] == cat])
    in_blue_ocean = len(blue_ocean[blue_ocean['primary_category'] == cat])
    pct = (in_blue_ocean / total) * 100
    gap_data.append({'Category': cat, 'Total': total, 'In Blue Ocean': in_blue_ocean, 'Blue Ocean %': pct})

gap_df = pd.DataFrame(gap_data)
gap_df = gap_df.sort_values('Blue Ocean %', ascending=True)
print(gap_df.to_string(index=False))

In [ ]:
# ============================================
# STORY 4: FINAL RECOMMENDATION
# ============================================

# Get Confectionery Blue Ocean stats
conf_blue = blue_ocean[blue_ocean['primary_category'] == 'Confectionery']

avg_protein = conf_blue['proteins_100g'].mean()
avg_sugar = conf_blue['sugars_100g'].mean()

print("=" * 65)
print("KEY INSIGHT — MARKET GAP RECOMMENDATION")
print("=" * 65)
print(f"""
Based on the data, the biggest market opportunity is in 
CONFECTIONERY, specifically targeting products with at least 
{avg_protein:.0f}g of protein and less than {avg_sugar:.0f}g of sugar per 100g.

Only 3.8% of the 6,022 confectionery products analyzed 
meet the High Protein + Low Sugar criteria — meaning 
96.2% of the confectionery market is still stuck in the 
'Sugar Trap'. This is the Blue Ocean.
""")
print("=" * 65)

In [ ]:
# ============================================
# BONUS STORY: THE HIDDEN GEM
# ============================================

# Get high protein confectionery products that have ingredients listed
conf_high_protein = df_final[
    (df_final['primary_category'] == 'Confectionery') &
    (df_final['proteins_100g'] > 13) &
    (df_final['ingredients_text'].notna())
]

print(f"High protein confectionery products with ingredients: {len(conf_high_protein)}")

# Common protein sources to search for
protein_sources = [
    'whey', 'soy', 'peanut', 'almond', 'cashew',
    'milk protein', 'egg', 'casein', 'pea protein',
    'rice protein', 'hemp', 'collagen', 'gelatin',
    'nut', 'seed', 'quinoa', 'chickpea', 'lentil'
]

# Count how many products contain each protein source
print("\nProtein Source Analysis:")
print("-" * 40)

source_counts = {}
for source in protein_sources:
    count = conf_high_protein['ingredients_text'].str.lower().str.contains(source, na=False).sum()
    source_counts[source] = count

# Sort by count
source_counts = dict(sorted(source_counts.items(), key=lambda x: x[1], reverse=True))

for source, count in source_counts.items():
    pct = (count / len(conf_high_protein)) * 100 if len(conf_high_protein) > 0 else 0
    print(f"  {source.title()}: {count} products ({pct:.1f}%)")

# Top 3
top3 = list(source_counts.keys())[:3]
print(f"\n🏆 TOP 3 PROTEIN SOURCES IN HIGH PROTEIN CONFECTIONERY:")
for i, source in enumerate(top3, 1):
    print(f"  {i}. {source.title()} — found in {source_counts[source]} products")

In [ ]:
# ============================================
# CANDIDATE'S CHOICE: MARKET OPPORTUNITY SCORE
# ============================================

import numpy as np

print("Building Market Opportunity Score...\n")

opportunity_data = []

for cat in df_final['primary_category'].unique():
    cat_df = df_final[df_final['primary_category'] == cat]
    cat_blue = blue_ocean[blue_ocean['primary_category'] == cat]
    
    total = len(cat_df)
    in_blue_ocean = len(cat_blue)
    gap_pct = 100 - (in_blue_ocean / total * 100)  # Higher = bigger gap
    avg_protein = cat_df['proteins_100g'].mean()
    avg_sugar = cat_df['sugars_100g'].mean()
    
    # Market Opportunity Score Formula:
    # Gap % × log(market size) — rewards big markets with big gaps
    opportunity_score = (gap_pct / 100) * np.log(total) * 10
    
    opportunity_data.append({
        'Category': cat,
        'Total Products': total,
        'Healthy Products': in_blue_ocean,
        'Gap %': round(gap_pct, 1),
        'Avg Protein (g)': round(avg_protein, 1),
        'Avg Sugar (g)': round(avg_sugar, 1),
        'Opportunity Score': round(opportunity_score, 1)
    })

opportunity_df = pd.DataFrame(opportunity_data)
opportunity_df = opportunity_df.sort_values('Opportunity Score', ascending=False)
opportunity_df = opportunity_df.reset_index(drop=True)
opportunity_df.index += 1  # Start ranking from 1

print("=" * 75)
print("MARKET OPPORTUNITY SCORECARD")
print("=" * 75)
print(opportunity_df.to_string())
print("\n💡 Score = Gap Size × Market Size (higher = bigger opportunity)")
print(f"\n🏆 #1 Opportunity: {opportunity_df.iloc[0]['Category']} with score {opportunity_df.iloc[0]['Opportunity Score']}")

In [ ]:
# Export a clean sample for the dashboard
df_final.to_csv('../data/dashboard_data.csv', index=False)
print(f"Exported {len(df_final)} rows")
print("File saved as dashboard_data.csv")